In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.read_csv("assesments.csv")

In [ ]:
pd.read_csv("courses.csv")

In [ ]:
# Check data types
print("Data types before cleaning:")
print(assesments.dtypes)

# Convert numeric columns to numeric data types
assesments["score"] = pd.to_numeric(assesments["score"])
assesments["attendance_pct"] = pd.to_numeric(assesments["attendance_pct"])

print("\nData types after validation:")
print(assesments.dtypes)

In [ ]:
# Check row count before removing duplicates
print("Rows before duplicate removal:", len(assesments))

# Remove exact duplicate rows
assesments = assesments.drop_duplicates().copy()

# Check row count after removing duplicates
print("Rows after duplicate removal:", len(assesments))

In [ ]:
# Merge assesments with courses using a LEFT JOIN
merged_data = assesments.merge(
    courses,
    on="course_id",
    how="left",
    validate="many_to_one"
)

# Display merged data
display(merged_data.head())

# Check number of rows
print("Rows after merge:", len(merged_data))

In [ ]:
# Validate merged data

assert len(merged_data) == 12, "Expected 12 rows after cleaning and merge"
assert merged_data["department"].notna().all(), "Unmatched course_id found"

print("All validation checks passed!")
print("Rows:", len(merged_data))
print("Unmatched course IDs:", merged_data["department"].isna().sum())

In [ ]:
# Create pass_flag
# Score >= 50 = Pass (1), otherwise Fail (0)

merged_data["pass_flag"] = (merged_data["score"] >= 50).astype(int)

display(merged_data[["assessment_id", "course_id", "score", "pass_flag"]])

In [ ]:
# Create department summary

department_summary = (
    merged_data.groupby("department")
    .agg(
        assessments=("assessment_id", "count"),
        average_score=("score", "mean"),
        pass_count=("pass_flag", "sum"),
        pass_rate=("pass_flag", "mean")
    )
    .reset_index()
)

department_summary["pass_rate"] = department_summary["pass_rate"] * 100

display(department_summary)

In [ ]:
# Create course-wise summary

course_summary = (
    merged_data.groupby(["course_id", "course"])
    .agg(
        assessments=("assessment_id", "count"),
        average_score=("score", "mean"),
        pass_rate=("pass_flag", "mean")
    )
    .reset_index()
)

course_summary["pass_rate"] = course_summary["pass_rate"] * 100

# Find course with the lowest pass rate
lowest_course = course_summary.loc[course_summary["pass_rate"].idxmin()]

display(course_summary)

print("\nCourse with lowest pass rate:")
display(lowest_course)

In [ ]:
# Create monthly average score summary

monthly_summary = (
    merged_data.groupby("month", sort=False)["score"]
    .mean()
    .reindex(["Jan", "Feb", "Mar"])
    .reset_index()
)

display(monthly_summary)

# Create chart
plt.figure(figsize=(7, 4))

plt.plot(
    monthly_summary["month"],
    monthly_summary["score"],
    marker="o"
)

plt.title("Monthly Average Score")
plt.xlabel("Month")
plt.ylabel("Average Score")
plt.grid(True, alpha=0.3)

plt.tight_layout()

# Save chart
plt.savefig("../outputs/python_chart.png", dpi=150)

plt.show()

In [ ]:
from pathlib import Path

# Create outputs folder if it does not exist
Path("../outputs").mkdir(exist_ok=True)

# Save the chart
plt.savefig("../outputs/python_chart.png", dpi=150)

print("Chart saved successfully!")

In [ ]:
# Export final cleaned data
merged_data.to_csv("../outputs/clean_data.csv", index=False)

# Export department summary
department_summary.to_csv("../outputs/python_summary.csv", index=False)

print("Files exported successfully!")
print("1. clean_data.csv")
print("2. python_summary.csv")